# Bayesian MLP, 24 lags + business-hour dummy

Thesis experiment with the `blitz` Bayes-by-backprop layers. Needs `blitz-bayesian-pytorch`.

In [ ]:
import pandas as pd
import numpy as np
from math import sqrt
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from blitz.modules import BayesianLinear
from blitz.utils import variational_estimator

In [ ]:
from energyforecast.data import load_dataset

data = load_dataset("../../data")

## Model

In [ ]:
def plot_model_rmse_and_loss(train):
    #evaluating train and validation accuracies and losses
    train_loss = train
    #val_loss = val
    #visualizing epochs vs. train and validation accuracies and losses
    plt.figure(figsize=(20, 10))
    plt.plot(train_loss, label='Training Loss')
    #plt.plot(val_loss, label='Validation Loss')
    plt.legend()
    plt.title('Epochs / Training Loss')
    plt.show()

def plot_preds_vs_actual(true, preds):
    plt.figure(figsize=(12,6))
    plt.plot(true, label='Real')
    plt.plot(preds, label='Predicted')
    plt.title('Predicted vs Real Values')
    plt.title('Actual vs Predicted Values')
    plt.xlabel('Time')
    plt.ylabel('Total Aggregated')
    plt.legend()
    plt.show()

In [ ]:
@variational_estimator
class BayesianRegressor(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.layer = nn.Sequential(
            BayesianLinear(input_dim, 10, prior_sigma_1=0.01, prior_sigma_2=0.01, prior_pi=1, posterior_mu_init=0),
            nn.ReLU(),
            BayesianLinear(10, 10),
            nn.ReLU(),
    #BayesianLinear(10, 10),
    #nn.ReLU(),
    #BayesianLinear(5, 5),
    #nn.ReLU(),
            BayesianLinear(10, output_dim)
        )

    #self.blinear1 =
    #self.blinear1 =
    #self.blinear3 =

    #def forward(self, x):
    #    x_ = self.blinear1(x)
    #    x_ = F.relu(x_)
    #    x_ = self.blinear2(x)
    #    x_ = F.relu(x_)
    #    return self.blinear3(x_)
    def forward(self, x):
        x_ = self.layer(x)
        return x_

In [ ]:
n_train = 35064
features = ['total_aggregated','business_hour']
feature_array = data[features].values

# Fit Scaler only on Training target values
scaler = StandardScaler()
scaler.fit(feature_array[:n_train, 0].reshape(-1,1))

# Transform both Training and Test data
scaled_feature = scaler.transform(feature_array[:, 0].reshape(-1, 1))

scaled_array = np.copy(feature_array)
scaled_array[:, 0] = scaled_feature.flatten()

# Create a DataFrame from the observations
df = pd.DataFrame(scaled_array, columns=['total_aggregated', 'biz'])

# Create 24 lags and store them as new columns in the DataFrame
for i in range(1, 25):
    df[f'lag_{i}'] = df['total_aggregated'].shift(i)

# Interpolate missing values
df = df.bfill()

# Define the number of train and test observations
# Separate the lags (features) and the observation (target)
X = df.drop('total_aggregated', axis=1)
y = df['total_aggregated']

# Separate the features into training and test sets
X_train = X[:n_train]
X_test = X[n_train:]

y_train, y_test = y[:n_train], y[n_train:]

X_train3, X_test3, y_train3, y_test3 = torch.tensor(X_train.values).float(), torch.tensor(X_test.values).float(), torch.tensor(y_train.values).float(), torch.tensor(y_test.values).float()

In [ ]:
def evaluate_regression(regressor,
                        X,
                        y,
                        samples = 200,
                        std_multiplier = 3):
    preds = [regressor(X) for i in range(samples)]
    preds = torch.stack(preds)
    means = preds.mean(axis=0)
    y_pred = scaler.inverse_transform(means.detach().numpy())

    stds = preds.std(axis=0)

    ci_upper = means + (std_multiplier * stds)
    upper_inv = scaler.inverse_transform(ci_upper.detach().numpy())

    ci_lower = means - (std_multiplier * stds)
    lower_inv = scaler.inverse_transform(ci_lower.detach().numpy())

    #y_true = scaler.inverse_transform(y.detach().numpy())

    #ic_acc = (ci_lower < y_vec) == (ci_upper > y_vec)
    #ic_acc = ic_acc.mean()
    return y_pred, upper_inv, lower_inv, stds # means.float(), ic_acc, (ci_upper >= y).float(), (ci_lower <= y).float(), stds.float()

In [ ]:
regressor3 = BayesianRegressor(25,1)
optimizer = optim.Adam(regressor3.parameters(), lr=0.01)
criterion = torch.nn.MSELoss()

ds_train = torch.utils.data.TensorDataset(X_train3, y_train3)
dataloader_train = torch.utils.data.DataLoader(ds_train, batch_size=168, shuffle=False)

ds_test = torch.utils.data.TensorDataset(X_test3, y_test3)
dataloader_test = torch.utils.data.DataLoader(ds_test, batch_size=168, shuffle=True)

In [ ]:
def train_model(regressor, dataloader, optimizer, criterion, num_epochs=3):
    regressor.train()
    for epoch in range(num_epochs):  # epochs per forecast period
        elbo = 0.0
        mse = 0.0
        kl_div = 0.0
        for i, (datapoints, labels) in enumerate(dataloader):
            optimizer.zero_grad()
            loss = regressor.sample_elbo_detailed_loss(inputs=datapoints,
                                                       labels=labels.unsqueeze(1),
                                                       criterion=criterion,
                                                       sample_nbr=5,
                                                       complexity_cost_weight=1./X_train.shape[0])
            loss[1].backward()
            optimizer.step()
            elbo += loss[1]
            mse += loss[2]
            kl_div += loss[3]
        elbo /= len(dataloader)
        mse /= len(dataloader)
        kl_div /= len(dataloader)
        print(f"Epoch {epoch+1} - ELBO: {elbo:.6f}, MSE: {mse}, KL DIV: {kl_div}")

In [ ]:
n_train = 24*365*3  # training window length
n_forecast_steps = 24*7  # forecast horizon

history_X = X_train3
history_y = y_train3

predictions3 = []  # forecasts
errors3 = []  # RMSE per period
days = 1  # period counter
uppers3 = []  # upper interval bound
lowers3 = []  # lower interval bound
#deviations = []

for i in range(0, len(X_test), n_forecast_steps):
    t_end = i + n_forecast_steps  # forecast the block [i, i + n_forecast_steps)

    if t_end > len(X_test):
        t_end = len(X_test)
    X_test_block = X_test3[i:t_end]
    y_test_block = y_test3[i:t_end]

    ds_train = torch.utils.data.TensorDataset(X_train3, y_train3)
    dataloader_train = torch.utils.data.DataLoader(ds_train, batch_size=168, shuffle=False)

    ds_test = torch.utils.data.TensorDataset(X_test3, y_test3)
    dataloader_test = torch.utils.data.DataLoader(ds_test, batch_size=168, shuffle=True)

    # Train the model
    train_model(regressor3, dataloader_train, optimizer, criterion)  # fit on the current history

    y_pred, ci_upper, ci_lower, stds = evaluate_regression(regressor3, X_test_block, y_test_block)  # forecast the next block

    predictions3.extend(y_pred)
    uppers3.extend(ci_upper)
    lowers3.extend(ci_lower)

    history_X = torch.cat([history_X[24:], X_test_block])  # slide the history forward by one block:
    history_y = torch.cat([history_y[24:], y_test_block])  # drop the oldest hours, append the observed ones

In [ ]:
y_true = np.array(data.total_aggregated[-len(predictions3):])
up = np.array(uppers3).flatten()
lo = np.array(lowers3).flatten()
u_upper = up >= y_true
o_lower = lo <= y_true
total = (u_upper == o_lower)

#
print("{} our predictions are in our confidence interval".format(np.mean(total)))

In [ ]:
truth = []
for i in y_true:
    truth.append(float(i))

In [ ]:
res = pd.DataFrame()
res['y_true'] = truth
res['y_pred'] = predictions3
res['upper_ci_bound'] = up
res['lower_ci_bound'] = lo

mse = sqrt(np.mean((res.y_pred - res.y_true)**2))
print('RMSE:', mse)

In [ ]:
#plot_preds_vs_actual(res.y_true, res.y_pred)
plt.figure(figsize=(20,10))
plt.title("Total aggregated load", color="black")

plt.plot(res.index,
         res.y_true,
         color='Blue',
         label="Real")

plt.plot(res.index,
         res.y_pred,
         label="Prediction",
         color="red")

plt.fill_between(x=res.index,
                 y1=res.upper_ci_bound,
                 y2=res.lower_ci_bound,
                 color='orange',
                 label="Confidence interval",
                 alpha=1)

plt.legend()